# 윈도우 라벨 매핑

CloudTrail 이벤트를 **5개씩 윈도우로 분할**할 때, 각 윈도우에 전술(tactic)·기법(technique) 라벨을 어떻게 붙일지 정리한다.

- 이벤트 단위 라벨: 각 이벤트는 `benign`이거나 하나의 (전술, 기법) 라벨을 가진다.
- 윈도우 단위 라벨: 윈도우 안의 5개 이벤트 라벨을 모아 윈도우 라벨을 만든다.

## 방식 1. 공격 이벤트가 하나라도 있으면 공격 라벨

윈도우 안에 **공격 이벤트가 하나라도 있으면** 그 윈도우 전체를 공격으로 라벨링한다.

- 공격 이벤트가 없으면 → `benign`
- 공격 이벤트가 1개 이상이면 → 그 이벤트의 전술·기법 라벨을 윈도우 라벨로 사용
- 서로 다른 기법이 여러 개 섞여 있으면 → 등장한 기법(과 전술)을 모두 포함하는 **멀티라벨**

예시 (윈도우 크기 5):

| 윈도우 | 이벤트 라벨 | 윈도우 라벨 |
|---|---|---|
| w0 | benign ×5 | benign |
| w1 | benign ×4, T1078 | TA0001 / T1078 |
| w2 | T1078, benign ×3, T1530 | {TA0001, TA0009} / {T1078, T1530} |

**장점**
- 공격을 놓치지 않는다 (recall 관점에서 유리).
- 규칙이 단순해서 구현과 설명이 쉽다.

**단점**
- 공격 이벤트가 1개뿐이고 나머지 4개가 정상이어도 공격으로 라벨링되어, 윈도우 텍스트 대부분이 정상 행위인 노이즈 라벨이 생긴다.
- 슬라이딩 윈도우(stride < 5)를 쓰면 같은 공격 이벤트가 여러 윈도우에 공격 라벨을 퍼뜨려 공격 윈도우 수가 부풀려진다.

## 예시 코드

In [1]:
import pandas as pd

# 이벤트 단위 라벨 예시 (benign이면 tactic/technique/attack_id = None)
# attack_id: 한 번의 기법 실행을 이루는 이벤트 묶음 ID (방식 2에서 사용)
events = pd.DataFrame({
    "eventName": [
        "DescribeInstances", "ListBuckets", "GetCallerIdentity", "DescribeRegions", "ListUsers",
        "DescribeInstances", "ListBuckets", "GetCallerIdentity", "DescribeRegions", "ConsoleLogin",
        "ConsoleLogin", "ListUsers", "DescribeRegions", "ListBuckets", "GetObject",
    ],
    "tactic":    [None]*9 + ["TA0001", "TA0001", None, None, None, "TA0009"],
    "technique": [None]*9 + ["T1078",  "T1078",  None, None, None, "T1530"],
    "attack_id": [None]*9 + ["a1",     "a1",     None, None, None, "a2"],
})
events

,eventName,tactic,technique,attack_id
0,DescribeInstances,NaN,NaN,NaN
1,ListBuckets,NaN,NaN,NaN
2,GetCallerIdentity,NaN,NaN,NaN
3,DescribeRegions,NaN,NaN,NaN
4,ListUsers,NaN,NaN,NaN
5,DescribeInstances,NaN,NaN,NaN
6,ListBuckets,NaN,NaN,NaN
7,GetCallerIdentity,NaN,NaN,NaN
8,DescribeRegions,NaN,NaN,NaN
9,ConsoleLogin,TA0001,T1078,a1


In [2]:
def make_windows(df, size=5, stride=5):
    """이벤트를 size개씩 묶는다. stride < size이면 슬라이딩 윈도우."""
    return [df.iloc[i:i + size] for i in range(0, len(df) - size + 1, stride)]


def label_any_attack(window):
    """방식 1: 공격 이벤트가 하나라도 있으면 등장한 전술·기법을 모두 윈도우 라벨로."""
    attack = window.dropna(subset=["technique"])
    if attack.empty:
        return {"is_attack": False, "tactics": [], "techniques": []}
    return {
        "is_attack": True,
        "tactics": sorted(attack["tactic"].unique()),
        "techniques": sorted(attack["technique"].unique()),
    }

In [3]:
rows = []
for i, w in enumerate(make_windows(events, size=5, stride=5)):
    rows.append({"window": f"w{i}", "n_attack_events": w["technique"].notna().sum(), **label_any_attack(w)})
pd.DataFrame(rows)

,window,n_attack_events,is_attack,tactics,techniques
0,w0,0,False,[],[]
1,w1,1,True,[TA0001],[T1078]
2,w2,2,True,"[TA0001, TA0009]","[T1078, T1530]"


슬라이딩 윈도우(stride=1)에서는 같은 공격 이벤트가 여러 윈도우에 걸쳐 공격 라벨을 만든다.

In [4]:
rows = []
for i, w in enumerate(make_windows(events, size=5, stride=1)):
    rows.append({"window": f"w{i}", "n_attack_events": w["technique"].notna().sum(), **label_any_attack(w)})
sliding = pd.DataFrame(rows)
print("공격 윈도우 비율:", sliding["is_attack"].mean().round(2))
sliding

공격 윈도우 비율: 0.55


,window,n_attack_events,is_attack,tactics,techniques
0,w0,0,False,[],[]
1,w1,0,False,[],[]
2,w2,0,False,[],[]
3,w3,0,False,[],[]
4,w4,0,False,[],[]
5,w5,1,True,[TA0001],[T1078]
6,w6,2,True,[TA0001],[T1078]
7,w7,2,True,[TA0001],[T1078]
8,w8,2,True,[TA0001],[T1078]
9,w9,2,True,[TA0001],[T1078]


## 방식 2. 기법 이벤트 전체가 윈도우 안에 있어야 공격 라벨

하나의 기법 실행(`attack_id`)을 이루는 공격 이벤트가 **잘리지 않고 모두 윈도우 안에 들어 있을 때만** 그 전술·기법을 윈도우 라벨로 붙인다.

- 윈도우 안에 완전히 포함된 기법이 **하나라도 있으면** → 공격 (포함된 기법들의 전술·기법이 라벨, 여러 개면 멀티라벨)
- 완전히 포함된 기법이 **하나도 없으면** → `benign`
  - 공격 이벤트가 윈도우 경계에 걸쳐 일부만 들어 있는 경우도 `benign`으로 본다.

예시 (윈도우 크기 5, 기법 a1 = T1078 이벤트 2개가 w1·w2 경계에 걸침, a2 = T1530 이벤트 1개):

| 윈도우 | 이벤트 라벨 | 방식 1 | 방식 2 |
|---|---|---|---|
| w0 | benign ×5 | benign | benign |
| w1 | benign ×4, a1 일부 | TA0001 / T1078 | benign (a1이 잘림) |
| w2 | a1 일부, benign ×3, a2 | {TA0001, TA0009} / {T1078, T1530} | TA0009 / T1530 |

**장점**
- 라벨이 붙은 윈도우에는 그 기법의 행위가 온전히 들어 있어, 윈도우 텍스트와 라벨이 잘 맞는다 (라벨 정밀도 높음).
- 슬라이딩 윈도우에서도 기법 하나가 공격 라벨을 퍼뜨리는 범위가 줄어든다.

**단점**
- 이벤트 수가 5개보다 많은 기법은 어떤 윈도우에도 완전히 들어갈 수 없어 **영원히 공격으로 라벨링되지 않는다.** 윈도우 크기를 정하기 전에 기법별 이벤트 수 분포를 확인해야 한다.
- 경계에 걸친 공격 이벤트가 있는 윈도우가 `benign`으로 들어가, 정상 데이터에 공격 행위가 섞인다.
- 기법 실행 단위(`attack_id`)를 이벤트에 미리 붙여 두어야 한다.

In [5]:
def label_fully_contained(window, all_events):
    """방식 2: 이벤트 전체가 윈도우 안에 들어온 기법만 윈도우 라벨로."""
    total = all_events["attack_id"].value_counts()   # 기법 실행별 전체 이벤트 수
    in_win = window["attack_id"].value_counts()      # 윈도우 안에 들어온 이벤트 수
    full_ids = [a for a, n in in_win.items() if n == total[a]]
    attack = window[window["attack_id"].isin(full_ids)]
    if attack.empty:
        return {"is_attack": False, "tactics": [], "techniques": []}
    return {
        "is_attack": True,
        "tactics": sorted(attack["tactic"].unique()),
        "techniques": sorted(attack["technique"].unique()),
    }

방식 1과 방식 2를 같은 윈도우에서 비교한다.

In [6]:
def compare(df, size=5, stride=5):
    rows = []
    for i, w in enumerate(make_windows(df, size=size, stride=stride)):
        m1, m2 = label_any_attack(w), label_fully_contained(w, df)
        rows.append({
            "window": f"w{i}",
            "m1_techniques": m1["techniques"] or "benign",
            "m2_techniques": m2["techniques"] or "benign",
        })
    return pd.DataFrame(rows)

compare(events, stride=5)

,window,m1_techniques,m2_techniques
0,w0,benign,benign
1,w1,[T1078],benign
2,w2,"[T1078, T1530]",[T1530]


In [7]:
# 슬라이딩 윈도우(stride=1)
cmp = compare(events, stride=1)
print("공격 윈도우 수 — 방식 1:", (cmp["m1_techniques"] != "benign").sum(),
      "/ 방식 2:", (cmp["m2_techniques"] != "benign").sum())
cmp

공격 윈도우 수 — 방식 1: 6 / 방식 2: 5


,window,m1_techniques,m2_techniques
0,w0,benign,benign
1,w1,benign,benign
2,w2,benign,benign
3,w3,benign,benign
4,w4,benign,benign
5,w5,[T1078],benign
6,w6,[T1078],[T1078]
7,w7,[T1078],[T1078]
8,w8,[T1078],[T1078]
9,w9,[T1078],[T1078]


In [8]:
# 기법 실행별 이벤트 수: 윈도우 크기(5)보다 큰 기법은 방식 2에서 라벨이 붙지 않는다
sizes = events.dropna(subset=["attack_id"]).groupby(["attack_id", "technique"]).size().rename("n_events")
sizes.to_frame().assign(fits_in_window=sizes <= 5)

,,n_events,fits_in_window
attack_id,technique,,
a1,T1078,2,True
a2,T1530,1,True


## 실제 데이터에 적용 (방식 1)

`preprocess.ipynb`가 만든 이벤트를 윈도우로 나누고 라벨을 붙여 저장한다. 라벨 규칙은 `labels/README.md`.

- 정책: 크기 5, stride 1, 방식 1 (공격 이벤트가 하나라도 있으면 공격)
- 입력: `data/interim/events_A.csv`(압축 안 함), `events_B.csv`(반복 압축)
- 출력: `data/interim/windows_{A,B}_size5_stride1.csv` (다음 단계 `04_split_v2.ipynb`가 읽음)
- 세션(원본 파일)마다 시간순으로 정렬해서 따로 분할한다. 세션끼리 이어 붙이지 않는다.
- 위 예시의 `make_windows`와 달리, 끝에 남는 5개 미만 이벤트도 윈도우 하나로 남긴다 (이벤트가 1~4개뿐인 Stratus 실행이 사라지지 않게).
- `session`, 시간, `ip`, `ip_kind`, 라벨은 **정렬·분할까지만** 쓰고 모델 입력에는 넣지 않는다.

**윈도우 라벨**

| 열 | 내용 |
|---|---|
| `m1_is_attack` | 공격 이벤트가 하나라도 있으면 True |
| `m1_tactics` | 공격 이벤트들의 MITRE 전술 **각각**의 합집합 (L1, L5) |
| `m1_techniques` | 공격 이벤트들의 기법 **+ 상위 기법** 합집합 (L3, L5) |
| `u_tactics`, `u_techniques` | 같은 내용(signature) 윈도우 묶음 전체의 라벨 합집합. 정상 윈도우가 섞이면 `BENIGN`도 포함 (L6) |
| `is_first` | 묶음의 대표 윈도우 (처음 나온 것) |
| `label_conflict` | 묶음 안 윈도우끼리 라벨이 다름 (u_*가 합집합으로 바뀐 경우) |
| `n_attack_events`, `n_benign_events` | 윈도우 안 공격 이벤트 수, 정상 이벤트 수 |
| `mixed` | 공격 이벤트와 정상 이벤트가 섞인 윈도우. **분석용**이며 라벨에는 반영하지 않는다 |

윈도우 안에 정상 이벤트가 섞여도 공격이 하나라도 있으면 공격 라벨만 붙인다 (`BENIGN`은 붙이지 않음). 어느 이벤트가 공격인지 알 수 있는 경우라서, 정상 이벤트는 배경일 뿐이다. 내용만으로 정상인지 공격인지 알 수 없는 경우(같은 내용 윈도우가 정상·공격 둘 다로 나옴)만 `u_*`에서 `BENIGN`을 함께 붙인다 (L6).

윈도우는 지우지 않고 표시만 한다.

In [9]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = ROOT / "data" / "interim"
INPUTS = {"A": INTERIM / "events_A.csv", "B": INTERIM / "events_B.csv"}
SIZE, STRIDE = 5, 1

# 윈도우 내용 비교에 쓰는 모델 입력 필드 (IP, 세션, 시간, 라벨은 제외)
SIGNATURE_COLS = ["eventSource", "eventName", "errorCode", "readOnly", "userType", "region_changed", "repeat_count"]


def load_events(path):
    events = pd.read_csv(path, keep_default_na=False, na_values={"attack_id": [""], "technique": [""]})
    events["is_attack"] = events["is_attack"].astype(str) == "True"
    return events.sort_values(["session", "eventTime", "seq"], kind="stable")


def make_windows_keep_tail(df, size=5, stride=1):
    """이벤트를 size개씩 묶고, 끝에 남는 size개 미만 이벤트도 윈도우로 남긴다."""
    starts = list(range(0, max(len(df) - size, 0) + 1, stride))
    if starts[-1] + size < len(df):
        starts.append(starts[-1] + stride)
    return [df.iloc[i:i + size] for i in starts]


def union(col):
    """'|'로 이어진 라벨들의 합집합."""
    return "|".join(sorted({t for ts in col for t in str(ts).split("|") if t}))


def label_window(window):
    """방식 1: 공격 이벤트가 하나라도 있으면, 그 이벤트들의 전술(각각)과 기법(+상위 기법)을 모두 윈도우 라벨로."""
    attack = window[window["is_attack"]]
    if attack.empty:
        return {"m1_is_attack": False, "m1_tactics": "", "m1_techniques": ""}
    return {"m1_is_attack": True, "m1_tactics": union(attack["tactics"]), "m1_techniques": union(attack["techniques"])}


def window_signature(window):
    """모델 입력 필드만으로 윈도우 내용을 문자열로 만든다.
    actor는 세션마다 번호가 다르므로 윈도우 안에서 처음 나온 순서로 다시 번호를 매긴다."""
    actor = pd.factorize(window["actor"])[0]
    ev = window[SIGNATURE_COLS].astype(str).agg("|".join, axis=1)
    return " > ".join(f"{e}|actor{a}" for e, a in zip(ev, actor))

In [10]:
def build_windows(events, size=5, stride=1):
    rows = []
    for session, g in events.groupby("session", sort=False):
        for i, w in enumerate(make_windows_keep_tail(g, size, stride)):
            rows.append({
                "session": session, "source": w["source"].iat[0], "window": i,
                "n_events": len(w), "n_attack_events": int(w["is_attack"].sum()),
                "n_benign_events": int((~w["is_attack"]).sum()),
                "mixed": bool(w["is_attack"].any() and (~w["is_attack"]).any()),   # 분석용, 라벨에는 반영 안 함
                "event_uids": "|".join(w["event_uid"]),
                "eventNames": " > ".join(w["eventName"]),
                "signature": window_signature(w),
                **label_window(w),
            })
    return pd.DataFrame(rows)


def mark_duplicates(win):
    """같은 signature 윈도우를 한 묶음으로 표시하고 묶음 라벨의 합집합(u_*)을 붙인다 (L6).
    정상과 공격이 섞인 묶음은 BENIGN과 공격 라벨을 함께 갖는다: 이 패턴만으로는 어느 쪽인지 알 수 없다."""
    win = win.copy()
    tac = win["m1_tactics"].where(win["m1_is_attack"], "BENIGN")
    tech = win["m1_techniques"].where(win["m1_is_attack"], "BENIGN")
    g = win.groupby("signature", sort=False)
    win["dup_group"] = g.ngroup()
    win["dup_count"] = g["signature"].transform("size")
    win["is_first"] = ~win["signature"].duplicated()
    win["u_tactics"] = tac.groupby(win["signature"]).transform(union)
    win["u_techniques"] = tech.groupby(win["signature"]).transform(union)
    win["label_conflict"] = tech.groupby(win["signature"]).transform("nunique") > 1
    return win


windows = {v: mark_duplicates(build_windows(load_events(path), SIZE, STRIDE)) for v, path in INPUTS.items()}
for v, w in windows.items():
    out = INTERIM / f"windows_{v}_size{SIZE}_stride{STRIDE}.csv"
    w.to_csv(out, index=False)
    print(f"saved {out.relative_to(ROOT)}: {len(w)} windows")

saved data/interim/windows_A_size5_stride1.csv: 28387 windows
saved data/interim/windows_B_size5_stride1.csv: 8183 windows


### A와 B 비교

- `합친 뒤`: 같은 내용 윈도우를 하나로 (`is_first`). 라벨은 묶음 합집합(`u_*`)
- `라벨 합집합`: 묶음 안 라벨이 달라서 합집합으로 바뀐 묶음 수
- `정상+공격`: 합집합에 `BENIGN`과 공격 라벨이 함께 있는 묶음 수
- `섞인 윈도우(mixed)`: 윈도우 안에 공격 이벤트와 정상 이벤트가 함께 있는 수 (라벨은 공격만)

In [11]:
def counts(w, union_labels=False):
    att = w["u_tactics"].str.contains("TA") if union_labels else w["m1_is_attack"]
    return {"윈도우": len(w), "정상": int((~att).sum()), "공격": int(att.sum())}


rows = {}
for v, w in windows.items():
    f = w[w["is_first"]]
    rows[f"{v} 전체"] = {**counts(w), "라벨 합집합": "", "정상+공격": "", "섞인 윈도우(mixed)": int(w["mixed"].sum())}
    rows[f"{v} 합친 뒤"] = {**counts(f, True), "라벨 합집합": int(f["label_conflict"].sum()),
                          "정상+공격": int((f["u_tactics"].str.contains("BENIGN") & f["u_tactics"].str.contains("TA")).sum()),
                          "섞인 윈도우(mixed)": int(f["mixed"].sum())}
pd.DataFrame(rows).T

,윈도우,정상,공격,라벨 합집합,정상+공격,섞인 윈도우(mixed)
A 전체,28387,19730,8657,,,229
A 합친 뒤,9287,5747,3540,84,2,220
B 전체,8183,4681,3502,,,174
B 합친 뒤,7733,4493,3240,46,0,174


전술·기법별 공격 윈도우 수 (같은 내용을 합친 뒤, 합집합 라벨 기준).

In [12]:
merged = {v: w[w["is_first"]] for v, w in windows.items()}
per_tactic = pd.DataFrame({v: w["u_tactics"].str.split("|").explode().value_counts() for v, w in merged.items()}).fillna(0).astype(int)
per_tech = pd.DataFrame({v: w["u_techniques"].str.split("|").explode().value_counts() for v, w in merged.items()}).fillna(0).astype(int)
display(per_tactic.sort_values("B", ascending=False))
per_tech.sort_values("B", ascending=False)

,A,B
u_tactics,,
BENIGN,5749,4493
TA0007,2837,2520
TA0040,493,453
TA0003,221,289
TA0004,220,288
TA0005,77,90
TA0001,77,90
TA0112,73,70
TA0009,26,49


,A,B
u_techniques,,
BENIGN,5749,4493
T1580,1870,1632
T1526,1040,932
T1496,471,432
T1496.001,464,429
T1098,156,210
T1078.004,77,90
T1078,77,90
T1098.001,57,62


같은 공격 실행이 A와 B에서 어떻게 보이는지 비교한다. 예: `ssm-send-command`는 도구가 결과를 기다리며 같은 조회를 50번 반복한다.

In [13]:
session = "stratus:aws.execution.ssm-send-command"
for v in ("A", "B"):
    w = merged[v]
    w = w[(w["session"] == session) & w["m1_is_attack"]]
    print(f"[{v}] 공격 윈도우 {len(w)}개")
    print(w["eventNames"].str.replace("DescribeInstanceInformation", "DII").head(6).to_string(index=False), "\n")

[A] 공격 윈도우 0개
Series([], ) 

[B] 공격 윈도우 1개
DII > SendCommand > DII > GetCommandInvocation 

